# NeuralMath — The Mathematical Anatomy of a Neural Network

**Updated English edition (2026).** This notebook follows the reviewed English pedagogical manuscript. It uses a 2–12–1 network, 64 synthetic training observations, eight illustrative test cases, a sigmoid output, squared-error loss, and stochastic gradient descent with derivatives written explicitly. The ten main code blocks correspond to the online article.

The observed classifications are an illustration, **not** evidence of generalization to real fruit measurements. The reference curve is used only to generate the synthetic data and is never supplied to the network.


## 1. Network model and loss sensitivities

For a standardized input \(\mathbf{x}^\ast\), the forward pass is
\[
\mathbf{z}^{(1)}=W^{(1)}\mathbf{x}^\ast+\mathbf{b}^{(1)},\quad
\mathbf h=\operatorname{ReLU}(\mathbf{z}^{(1)}),\qquad
z^{(2)}=W^{(2)}\mathbf h+c,\quad \widehat y=\sigma(z^{(2)}).
\]
For \(E_{\mathrm{SE}}=\tfrac12(\widehat y-y)^2\), backpropagation computes the *local loss sensitivities*
\[
\delta^{(2)}=(\widehat y-y)\widehat y(1-\widehat y),\qquad
\delta_j^{(1)}=\delta^{(2)}v_j\operatorname{ReLU}'(z_j^{(1)}).
\]
These sensitivities are derivatives, not physical error signals. Our computational convention is \(\operatorname{ReLU}'(0)=0\).


### Code 1 — Imports and dependencies


In [ ]:
"""Pedagogical example: fruit classification using a 2-12-1 network.
Dependencies: numpy and matplotlib. Run: python neural_network.py.
"""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


### Code 2 — ReLU activation


In [ ]:
def relu(s):
    """Hidden-layer activation: ReLU(s) = max(0,s)."""
    return np.maximum(0, s)


### Code 3 — Derivative of ReLU


In [ ]:
def relu_derivative(s):
    # ReLU is not differentiable at s=0; its computational derivative is set to 0.
    # This convention only defines the backpropagation step at that point.
    return (s > 0).astype(float)


### Code 4 — Sigmoid output


In [ ]:
def sigmoid(s):
    """Output activation, with values between 0 and 1."""
    return 1 / (1 + np.exp(-s))


## 2. Constructing synthetic data

The auxiliary diameter curve, in centimetres for mass \(m\) numerically expressed in grams, is
\[
d_0(m)=6.68+0.74\exp\!\left[-\left(\frac{m-155}{20}\right)^2\right]
+\frac{1.50}{1+\exp[-(m-202)/5]}.
\]
The four bands and deterministic perturbations are those given in Appendix A of the manuscript. This curve creates labels and locations only; it is **not** a learned decision boundary and never enters the training objective.


### Code 5 — Synthetic reference curve


In [ ]:
def reference_curve(m):
    # Artificial curve (cm): offset + Gaussian bump + logistic rise.
    # The constants were chosen to construct a nonlinear reference curve;
    # they are neither experimental measurements nor fitted parameters.
    return (6.68 + 0.74*np.exp(-((m-155)/20)**2)
            + 1.50/(1 + np.exp(-(m-202)/5)))


### Code 6 — Training and illustrative data


In [ ]:
def construct_data():
    # 16 masses in each of four bands: a1, a2, l1, l2.
    masses_1 = np.linspace(118, 212, 16)
    # Small deterministic offsets avoid identical masses.
    masses_2 = masses_1 + np.array([
        1.2, -1.1, 0.8, -1.3, 1.0, -0.8, 1.1, -1.0,
        1.3, -0.9, 0.9, -1.2, 1.0, -1.0, 0.8, -0.7
    ])
    k = np.arange(16)

    # Apples: two bands above the curve, with small variations in cm.
    a1 = np.column_stack([
        masses_1,
        reference_curve(masses_1) + 0.22 + 0.05*np.sin(0.8*k)
    ])
    a2 = np.column_stack([
        masses_2,
        reference_curve(masses_2) + 0.58 + 0.06*np.cos(0.45*k)
    ])

    # Oranges: two bands below the reference curve.
    l1 = np.column_stack([
        masses_1,
        reference_curve(masses_1) - (0.22 + 0.04*np.cos(0.7*k))
    ])
    l2 = np.column_stack([
        masses_2,
        reference_curve(masses_2) - (0.55 + 0.05*np.sin(0.5*k))
    ])

    X_train = np.vstack([a1, a2, l1, l2])
    y_train = np.array([1.0]*32 + [0.0]*32)

    # Eight illustrative examples; not an extensive statistical test.
    X_test = np.array([
        [121.0, 7.20], [135.0, 7.45],
        [152.0, 7.75], [171.0, 7.32],
        [121.0, 6.28], [152.0, 6.86],
        [189.0, 6.48], [208.0, 7.55]
    ])
    y_test = np.array([1, 1, 1, 1, 0, 0, 0, 0], dtype=float)
    return X_train, y_train, X_test, y_test


### The four synthetic bands (Figure 7)

The dashed reference curve constructs the sample geometry. The plot is descriptive, not a plot of the network's predictions.


In [ ]:
X_demo, y_demo, _, _ = construct_data()
xx = np.linspace(116, 214, 500)
fig, ax = plt.subplots(figsize=(9, 5))
mask = y_demo == 1
ax.scatter(X_demo[mask, 0], X_demo[mask, 1], label='Apple — training', marker='o', facecolors='white', edgecolors='#206d48')
ax.scatter(X_demo[~mask, 0], X_demo[~mask, 1], label='Orange — training', marker='s', facecolors='white', edgecolors='#b65a2e')
ax.plot(xx, reference_curve(xx), '--', color='#666666', label='Auxiliary curve (data generation only)')
ax.set(xlabel='Mass (g)', ylabel='Diameter (cm)', title='Synthetic examples used for training')
ax.legend()
plt.show()


### Code 7 — Prediction


In [ ]:
def predict(X, mean, std, W1, b1, W2, b2):
    # x_j^* = (x_j - mu_j)/sigma_j, using training-set statistics.
    Xn = (X - mean) / std
    # Forward pass in matrix notation (one row per fruit).
    h = relu(Xn @ W1.T + b1)
    return sigmoid(h @ W2 + b2)


### Code 8 — SGD with explicit backpropagation


In [ ]:
def train(Xn, y_train, eta=0.05, n_epochs=6000, seed=42):
    rng = np.random.default_rng(seed)
    # Small random weights break symmetry between neurons;
    # initial biases are zero. Scale 0.1 is a pedagogical choice.
    W1 = 0.1 * rng.standard_normal((12, 2))
    b1 = np.zeros(12)
    W2 = 0.1 * rng.standard_normal(12)
    b2 = 0.0
    history = []

    for epoch in range(n_epochs):
        # SGD: one update per fruit, shuffled in every epoch.
        for i in rng.permutation(len(Xn)):
            x, target = Xn[i], y_train[i]

            # Forward pass: hidden- and output-layer equations.
            z1 = W1 @ x + b1
            h = relu(z1)
            z2 = W2 @ h + b2
            y_hat = sigmoid(z2)

            # Backpropagation: delta2 = dE/dz2 (squared error).
            delta2 = (y_hat - target) * y_hat * (1 - y_hat)
            dW2, db2 = delta2 * h, delta2
            # delta1_j = delta2 * W2_j * ReLU'(z1_j).
            delta1 = (delta2 * W2) * relu_derivative(z1)
            dW1 = np.outer(delta1, x)
            db1 = delta1

            # Stochastic descent: theta <- theta - eta*dE/dtheta.
            # All gradients use the PREVIOUS parameter values.
            W1 -= eta * dW1
            b1 -= eta * db1
            W2 -= eta * dW2
            b2 -= eta * db2

        # J: mean training loss at the end of each epoch.
        h_all = relu(Xn @ W1.T + b1)
        y_pred = sigmoid(h_all @ W2 + b2)
        history.append(np.mean(0.5 * (y_train - y_pred)**2))

    return W1, b1, W2, b2, np.array(history)


### Code 9 — Decision-boundary plot


In [ ]:
def plot_results(X_train, y_train, X_test, y_test,
                  mean, std, W1, b1, W2, b2, save_path=None):
    # Level set y_hat=0.5: LEARNED decision boundary.
    masses = np.linspace(116, 214, 500)
    diameters = np.linspace(6.0, 8.85, 500)
    M, D = np.meshgrid(masses, diameters)
    grid = np.column_stack([M.ravel(), D.ravel()])
    P = predict(grid, mean, std, W1, b1, W2, b2).reshape(M.shape)

    fig, ax = plt.subplots(figsize=(9.2, 6.3))
    ax.contourf(M, D, P, levels=[0, 0.25, 0.50, 0.75, 1],
                colors=['#f3e0cb', '#ecd5c4', '#dbe7cf', '#cde3bf'],
                alpha=0.95)
    ax.contour(M, D, P, levels=[0.5], colors=['#5a1a8a'],
               linewidths=2.2)

    mask = y_train == 1.0
    ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='o',
               s=48, facecolors='white', edgecolors='black', linewidths=1.1)
    mask = y_train == 0.0
    ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='s',
               s=48, facecolors='white', edgecolors='black', linewidths=1.1)
    mask = y_test == 1.0
    ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='^',
               s=92, color='#1f77b4')
    mask = y_test == 0.0
    ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='x',
               s=110, color='#ff7f0e', linewidths=2.0)

    ax.set_xlabel('Mass (g)')
    ax.set_ylabel('Diameter (cm)')
    ax.set_title('Neural-network output after training')
    handles = [
        Line2D([0], [0], marker='o', markersize=9, markerfacecolor='white',
               markeredgecolor='black', linestyle='None',
               label='Apple - training'),
        Line2D([0], [0], marker='s', markersize=9, markerfacecolor='white',
               markeredgecolor='black', linestyle='None',
               label='Orange - training'),
        Line2D([0], [0], marker='^', markersize=9, color='#1f77b4',
               linestyle='None', label='Apple - test'),
        Line2D([0], [0], marker='x', markersize=10, color='#ff7f0e',
               linestyle='None', label='Orange - test'),
        Line2D([0], [0], color='#5a1a8a', linewidth=2.2,
               label='Learned boundary')
    ]
    ax.legend(handles=handles, loc='center left',
              bbox_to_anchor=(1.02, 0.5), frameon=True)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
    return fig, ax


### Code 10 — Run the complete experiment


In [ ]:
if __name__ == '__main__':
    X_train, y_train, X_test, y_test = construct_data()
    mean, std = X_train.mean(axis=0), X_train.std(axis=0)
    Xn = (X_train - mean) / std
    W1, b1, W2, b2, history = train(Xn, y_train)
    p_train = predict(X_train, mean, std, W1, b1, W2, b2)
    p_test = predict(X_test, mean, std, W1, b1, W2, b2)
    print('Final mean loss:', history[-1])
    print('Correct training predictions:', np.sum((p_train >= 0.5) == y_train), '/64')
    print('Correct illustrative test predictions:',
          np.sum((p_test >= 0.5) == y_test), '/8')
    plot_results(X_train, y_train, X_test, y_test,
                 mean, std, W1, b1, W2, b2)
    plt.show()


## 3. Worked numerical SGD step in a 2–3–1 network

For a labelled orange \(y=0\), let \(\mathbf{x}^\ast=(-1,-1)^\mathsf{T}\). The following independent calculation reproduces the manuscript's one-step example. The demonstration variables do not modify the trained 2–12–1 network.


In [ ]:
demo_x = np.array([-1.0, -1.0])
demo_y = 0.0
demo_W1 = np.array([[0.2, 0.1], [-0.3, 0.2], [0.1, -0.4]])
demo_b1 = np.array([0.5, 0.2, -0.5])
demo_v = np.array([0.5, -0.4, 0.3])
demo_c = 1.0
demo_z1 = demo_W1 @ demo_x + demo_b1
demo_h = relu(demo_z1)
demo_z2 = demo_v @ demo_h + demo_c
demo_p = sigmoid(demo_z2)
demo_delta2 = (demo_p - demo_y) * demo_p * (1 - demo_p)
demo_delta1 = demo_delta2 * demo_v * relu_derivative(demo_z1)
demo_eta = 0.05
print("Output:", round(float(demo_p), 6))
print("Output sensitivity:", round(float(demo_delta2), 6))
print("Hidden sensitivities:", np.round(demo_delta1, 6))
print("Updated output bias:", round(float(demo_c - demo_eta * demo_delta2), 6))
assert np.isclose(demo_p, 0.727108, atol=1e-6)


## 4. Squared error versus binary cross-entropy

For a sigmoid output and a target \(y\in\{0,1\}\),
\[
\frac{\partial E_{\mathrm{SE}}}{\partial z}
=(\widehat y-y)\widehat y(1-\widehat y),
\qquad
\frac{\partial E_{\mathrm{CE}}}{\partial z}=\widehat y-y.
\]
Cross-entropy avoids the extra sigmoid derivative factor in the **output-layer** sensitivity. It does not eliminate vanishing gradients elsewhere or guarantee faster optimization. The main experiment intentionally retains squared error.


In [ ]:
illustration_p, illustration_y, eta = 0.80, 0.0, 0.05
squared_loss = 0.5 * (illustration_p - illustration_y)**2
cross_entropy = -(illustration_y * np.log(illustration_p) + (1-illustration_y)*np.log(1-illustration_p))
grad_squared = (illustration_p - illustration_y)*illustration_p*(1-illustration_p)
grad_cross_entropy = illustration_p - illustration_y
print('Losses:', squared_loss, cross_entropy)
print('Score gradients:', grad_squared, grad_cross_entropy)
print('Isolated output-bias updates:', -eta*grad_squared, -eta*grad_cross_entropy)
assert np.allclose([squared_loss, grad_squared, grad_cross_entropy], [0.32, 0.128, 0.80])


## 5. Interpretation and reproducibility

Standardization uses training-set statistics only, which are reused without modification for the illustrative test cases. The program uses a fixed random seed, shuffled examples in each epoch, and explicit SGD. The synthetic data are not physical measurements, and neither their training accuracy nor the eight illustrative test outcomes support broad generalization claims. The PDF linked on NeuralMath.org is an earlier edition and will be updated separately.
